# Fine-tune Florence-2-large for OMR on A100 (PrIMuS semantic tokens)



**Goal:** image â†’ space-separated PrIMuS semantic token string using a **closed PrIMuS vocabulary** (~1.8k types, one token ID per semantic type).



| Item | Value |

|------|-------|

| Base model | `microsoft/Florence-2-large` |

| Decoder vocab | PrIMuS-only (`data/primus_semantic_tokens.txt`) |

| Data | `data/consolidated/` (~139k pairs) |

| Method | LoRA r=64 / alpha=128 / dropout=0.05 + bf16 |

| Learning rate | `1e-4` |

| Target GPU | NVIDIA A100 40 GB or 80 GB |



This is the **underfitting retry recipe**: twice the prior LoRA rank and learning rate to move the plateaued Florence-2-large run beyond its ~2.6–2.7 loss / ~13–15% golden accuracy regime.

See [`README_colab.md`](README_colab.md) for data upload and runtime notes.



**Before running:** select an A100 runtime, then upload `consolidated.zip`, `golden_set/`, and `scripts/primus_tokenizer.py`, `scripts/florence2_primus.py`, `data/primus_semantic_tokens.txt` to `MyDrive/omr/scripts/` (cell 2c).



**Fresh retry required:** the prior r32 adapter/checkpoints are incompatible with this r64 recipe. Outputs use the isolated directory `MyDrive/omr/florence2-large-handnew-r64-lr1e4/`.



**A100 tuning:** bf16 + TF32, fused AdamW, no gradient checkpointing, local-SSD data, persistent workers, and A100 40/80 GB batch plans. Florence-2 requires eager attention. The notebook uses Inductor compilation with CUDA Graph capture explicitly disabled, avoiding the LoRA gradient-accumulation overwrite bug while recovering compiled-kernel throughput.



**Benchmark flow:** golden eval before training â†’ LoRA fine-tune â†’ golden eval after.



In [ ]:
# @title 1) Mount Google Drive
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# @title 1b) Unzip consolidated data to LOCAL disk (fast — do NOT unzip onto Drive)
# ~139k small files unzip in a few minutes on Colab's local SSD, but can take HOURS
# (or stall) on the Google Drive mount, which heavily throttles small-file I/O.
# Training also reads them far faster from local disk. Upload a single
# consolidated.zip to MyDrive/omr/ — one file uploads reliably, whereas ~139k loose
# files crash the web uploader. See scripts/build_consolidated_zip.py to rebuild it.
import time
import zipfile
from pathlib import Path

ZIP_PATH = Path("/content/drive/MyDrive/omr/consolidated.zip")  # @param {type:"string"}
LOCAL_DATA_ROOT = Path("/content/omr_data")  # @param {type:"string"} local SSD (resets each session)

images_dir = LOCAL_DATA_ROOT / "consolidated" / "images"
if images_dir.is_dir() and any(images_dir.iterdir()):
    print(f"Local data already present: {images_dir} â€” skipping unzip")
elif ZIP_PATH.is_file():
    LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
    print(f"Extracting {ZIP_PATH} -> {LOCAL_DATA_ROOT} (local SSD)...")
    t0 = time.time()
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(LOCAL_DATA_ROOT)
    print(f"Extracted in {time.time() - t0:.0f}s -> {images_dir}")
else:
    print(f"No zip at {ZIP_PATH}.")
    print("Upload consolidated.zip to MyDrive/omr/ first (see scripts/build_consolidated_zip.py).")

In [ ]:
# @title 2) Install dependencies
# Pin numpy first — Colab ships numpy 2.x; install it before transformers to avoid ABI mismatches.
!pip install -q "numpy>=2.0.2,<2.4"
!pip install -q "transformers>=4.48.0,<5.0.0" accelerate peft datasets pillow tqdm sentencepiece einops timm huggingface_hub
# Colab ships torchao 0.10.x; recent peft rejects it — upgrade or remove
!pip install -q -U "torchao>=0.16.0" || pip uninstall -y torchao

# Allow Inductor, but CUDA Graphs will be disabled explicitly in configuration.
# The prior crash came from graph-captured LoRA outputs being reused across
# gradient-accumulation microsteps, not from Inductor compilation itself.
import os
os.environ.pop("TORCHDYNAMO_DISABLE", None)
os.environ.pop("ACCELERATE_DYNAMO_BACKEND", None)
print("TorchDynamo available; CUDA Graphs will remain disabled")

In [ ]:
# @title 2c) PrIMuS tokenizer utilities (upload scripts/ + vocab to Drive)

import sys

from pathlib import Path



SCRIPTS_CANDIDATES = [

    Path('/content/drive/MyDrive/omr/scripts'),

    Path('/content/small-language-model/scripts'),

    Path.cwd().parent / 'scripts',

]

for candidate in SCRIPTS_CANDIDATES:

    if (candidate / 'primus_tokenizer.py').is_file() and (candidate / 'florence2_primus.py').is_file():

        SCRIPTS_DIR = candidate

        break

else:

    raise FileNotFoundError(

        'Upload scripts/primus_tokenizer.py and scripts/florence2_primus.py to '

        '/content/drive/MyDrive/omr/scripts/'

    )



if str(SCRIPTS_DIR) not in sys.path:

    sys.path.insert(0, str(SCRIPTS_DIR))



from florence2_primus import (

    build_florence2_primus_processor,

    encode_semantic_labels,

    make_primus_logits_processor,

    resize_model_embeddings_for_primus,

)

from primus_tokenizer import PrimusTokenizer, split_semantic_tokens



VOCAB_CANDIDATES = [

    SCRIPTS_DIR / 'primus_semantic_tokens.txt',

    Path('/content/drive/MyDrive/omr/scripts/primus_semantic_tokens.txt'),

    Path('/content/small-language-model/data/primus_semantic_tokens.txt'),

]

for vp in VOCAB_CANDIDATES:

    if vp.is_file():

        PRIMUS_VOCAB_PATH = vp

        break

else:

    raise FileNotFoundError('Upload data/primus_semantic_tokens.txt to MyDrive/omr/scripts/')



print(f'SCRIPTS_DIR={SCRIPTS_DIR}')

print(f'PRIMUS_VOCAB_PATH={PRIMUS_VOCAB_PATH}')



In [ ]:
# @title 2b) Pin transformers + early Florence-2 compat (run before model load)
# Re-pin numpy before force-reinstalling transformers (prevents numpy._core.umath _center ImportError).
!pip install -q "numpy>=2.0.2,<2.4"
!pip install -q --force-reinstall --no-deps "transformers>=4.48.0,<5.0.0"
!pip install -q -U "torchao>=0.16.0" || pip uninstall -y torchao

import numpy as np
import transformers

print(f"numpy {np.__version__}")
print(f"transformers {transformers.__version__}")
if not np.__version__.startswith("2."):
    raise RuntimeError(
        f"Expected numpy 2.x (Colab default), got {np.__version__}. "
        "Runtime â†’ Restart session, then re-run cells 2 + 2b."
    )
if not transformers.__version__.startswith("4."):
    raise RuntimeError(
        f"Expected transformers 4.x, got {transformers.__version__}. "
        "Runtime â†’ Restart session, then re-run Install + this cell."
    )


def _early_florence2_compat_patch() -> None:
    """Patch config + tokenizer before any Florence-2 import (fixes additional_special_tokens)."""
    import transformers.configuration_utils as config_utils
    import transformers.tokenization_utils_base as tok_base

    _flag = "_florence2_tf5_compat_patched"
    if not getattr(config_utils.PretrainedConfig, _flag, False):
        _orig_init = config_utils.PretrainedConfig.__init__

        def _patched_init(self, *args, **kwargs):
            _orig_init(self, *args, **kwargs)
            if not hasattr(self, "forced_bos_token_id"):
                self.forced_bos_token_id = kwargs.get("forced_bos_token_id", None)

        config_utils.PretrainedConfig.__init__ = _patched_init
        setattr(config_utils.PretrainedConfig, _flag, True)

    ga_flag = "_florence2_tf5_getattr_patched"
    if not getattr(config_utils.PretrainedConfig, ga_flag, False):
        _orig_ga = config_utils.PretrainedConfig.__getattribute__

        def _patched_ga(self, key):
            if key == "forced_bos_token_id":
                try:
                    return _orig_ga(self, key)
                except AttributeError:
                    return None
            return _orig_ga(self, key)

        config_utils.PretrainedConfig.__getattribute__ = _patched_ga
        setattr(config_utils.PretrainedConfig, ga_flag, True)

    tok_flag = "_florence2_tf5_tok_getattr_patched"
    if getattr(tok_base.PreTrainedTokenizerBase, tok_flag, False):
        return
    if not hasattr(tok_base.PreTrainedTokenizerBase, "__getattr__"):
        print("WARN: PreTrainedTokenizerBase has no __getattr__ â€” tokenizer patch skipped")
        return

    _orig_getattr = tok_base.PreTrainedTokenizerBase.__getattr__

    def _patched_getattr(self, name):
        if name == "additional_special_tokens":
            try:
                return _orig_getattr(self, name)
            except AttributeError:
                try:
                    added = _orig_getattr(self, "added_tokens_decoder")
                except AttributeError:
                    return []
                return [
                    tok.content
                    for tok in added.values()
                    if getattr(tok, "special", False)
                ]
        return _orig_getattr(self, name)

    tok_base.PreTrainedTokenizerBase.__getattr__ = _patched_getattr
    setattr(tok_base.PreTrainedTokenizerBase, tok_flag, True)


_early_florence2_compat_patch()
print("Early Florence-2 compat patch applied (PreTrainedTokenizerBase.__getattr__)")


In [ ]:
# @title 3) Configuration â€” edit paths if needed
from __future__ import annotations

import json
import os
import re
import shutil
from pathlib import Path

# Florence-2 + PEFT fails only when Inductor captures CUDA Graphs across gradient-
# accumulation microsteps. Keep Dynamo available; CUDA Graph capture is disabled below.
os.environ.pop("TORCHDYNAMO_DISABLE", None)
os.environ.pop("ACCELERATE_DYNAMO_BACKEND", None)

import torch

# --- Paths (Colab / Drive) ---
DRIVE_ROOT = Path("/content/drive/MyDrive/omr")
# Raw training data lives on LOCAL disk (unzipped from consolidated.zip in cell 1b);
# the Drive mount is far too slow for ~139k small files. Golden set stays on Drive.
LOCAL_DATA_ROOT = Path("/content/omr_data")
CONSOLIDATED_DIR = LOCAL_DATA_ROOT / "consolidated"
IMAGES_DIR = CONSOLIDATED_DIR / "images"
GT_DIR = CONSOLIDATED_DIR / "ground_truth"
GOLDEN_DIR = DRIVE_ROOT / "golden_set"

# Isolate the r64/LR=1e-4 retry from incompatible r32 checkpoints and adapters.
DRIVE_OUTPUT_DIR = DRIVE_ROOT / "florence2-large-handnew-r64-lr1e4"
MANIFEST_PATH = DRIVE_OUTPUT_DIR / "manifest.json"
CHECKPOINT_DIR = DRIVE_OUTPUT_DIR / "checkpoints"

# --- Model ---
MODEL_ID = "microsoft/Florence-2-large"
TASK_PROMPT = "<OCR>"  # encoder task token (single ID in PrIMuS vocab)
USE_CONSTRAINED_DECODE = True  # mask logits to PrIMuS tokens at inference

# --- Split ---
MANIFEST_VERSION = 5  # PrIMuS closed-vocab; bumped for corrupt35 dataset (forces manifest rebuild)
GOLDEN_BENCHMARK_COUNT = 100
SEED = 1337

# --- Style mix (train sampler: 50% handwritten / 25% clean / 25% camera) ---
STYLE_WEIGHTS = {"handwritten": 0.50, "clean": 0.25, "camera": 0.25}
STYLE_WEIGHT_CLEAN_VS_CORRUPT = 0.50  # within each style, equal original vs corrupt when both exist

# --- Training (A100-optimized) ---
NUM_EPOCHS = 3
PER_DEVICE_BATCH_SIZE = 6       # A100 40 GB fallback; auto-tuned below
GRAD_ACCUM_STEPS = 8            # effective batch 48
AUTO_TUNE_COMPUTE = True        # distinguishes A100 40 GB vs 80 GB
# Safe Inductor path: compile the eager-attention model but prohibit CUDA Graph
# capture, which is the source of the LoRA gradient-accumulation overwrite error.
TORCH_COMPILE = True
# Underfitting retry recipe: 2x the previous LR and LoRA capacity after the
# large run plateaued near loss 2.6-2.7 and 13-15% golden token accuracy.
LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.03
MAX_LABEL_LENGTH = 96
WEIGHT_DECAY = 0.01
SAVE_STEPS = 500
LOGGING_STEPS = 50
SAVE_TOTAL_LIMIT = 4

# --- LoRA (underfitting retry: r64/alpha128; retain regularizing dropout) ---
LORA_R = 64
LORA_ALPHA = 128
LORA_DROPOUT = 0.05

# --- Resume / export ---
RESUME_FROM_CHECKPOINT = True   # auto-resume latest checkpoint-* if present
HF_REPO_ID = ""                 # e.g. "yourname/florence2-omr"; leave blank to skip Hub push
HF_TOKEN = os.environ.get("HF_TOKEN", "")  # or paste token here

# --- Golden benchmark eval ---
SKIP_BASELINE_EVAL = False  # fresh output dir has no cached baseline; recompute so post-train comparison has a delta
EVAL_GOLDEN_EVERY_EPOCH = True  # full golden benchmark after each training epoch
EVAL_GOLDEN_BEFORE_PATH = DRIVE_OUTPUT_DIR / "eval_golden_before.json"
EVAL_GOLDEN_AFTER_PATH = DRIVE_OUTPUT_DIR / "eval_golden_after.json"
EVAL_GOLDEN_EPOCH_DIR = DRIVE_OUTPUT_DIR / "eval_golden_epochs"
EVAL_GOLDEN_EPOCH_HISTORY_PATH = DRIVE_OUTPUT_DIR / "eval_golden_epoch_history.json"
EVAL_GOLDEN_EPOCH_WRITEUP_PATH = DRIVE_OUTPUT_DIR / "eval_golden_epochs.md"

# --- Optional WandB ---
USE_WANDB = False
WANDB_PROJECT = "florence2-omr"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE != "cuda":
    raise RuntimeError("This notebook requires an NVIDIA A100 runtime.")

_gpu_name = torch.cuda.get_device_name(0).lower()
_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
if "a100" not in _gpu_name:
    raise RuntimeError(
        f"Expected an A100, found {torch.cuda.get_device_name(0)}. "
        "Choose an A100 runtime or retune the batch plan before continuing."
    )
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("A100 bf16 support was not detected.")

USE_BF16 = True
USE_FP16 = False
GRADIENT_CHECKPOINTING = False  # recomputation is slower and unnecessary on A100
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import torch._dynamo
import torch._inductor.config as inductor_config
torch._dynamo.reset()
torch._dynamo.config.disable = False
# reduce-overhead mode enables CUDA Graphs; explicitly disable both graph paths and
# use normal Inductor kernels so gradient-accumulation microsteps own their outputs.
if hasattr(inductor_config.triton, "cudagraphs"):
    inductor_config.triton.cudagraphs = False
if hasattr(inductor_config.triton, "cudagraph_trees"):
    inductor_config.triton.cudagraph_trees = False
torch.set_float32_matmul_precision("high")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

# Florence-2-large adapts the vision tower as well as the language model. These
# conservative plans target effective batch 48 while leaving headroom for golden eval
# and transient allocations. If an 80 GB runtime has ample headroom, try batch 16 x 3.
_cpu = os.cpu_count() or 4
if AUTO_TUNE_COMPUTE:
    if _vram_gb >= 70:      # A100 80 GB
        PER_DEVICE_BATCH_SIZE, GRAD_ACCUM_STEPS = 12, 4
    elif _vram_gb >= 38:    # A100 40 GB
        PER_DEVICE_BATCH_SIZE, GRAD_ACCUM_STEPS = 6, 8
    else:
        raise RuntimeError(f"A100 detected but only {_vram_gb:.1f} GB VRAM is visible.")
DATALOADER_WORKERS = min(12, _cpu)

DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | VRAM: {props.total_memory / 1e9:.1f} GB")
print(
    f"Compute plan: batch={PER_DEVICE_BATCH_SIZE} x accum={GRAD_ACCUM_STEPS} "
    f"(effective {PER_DEVICE_BATCH_SIZE * GRAD_ACCUM_STEPS}) | "
    f"grad_ckpt={GRADIENT_CHECKPOINTING} | workers={DATALOADER_WORKERS} | "
    f"precision={'bf16' if USE_BF16 else 'fp16' if USE_FP16 else 'fp32'} | compile={TORCH_COMPILE}"
)
print(f"Images dir exists: {IMAGES_DIR.is_dir()}")
print(f"GT dir exists: {GT_DIR.is_dir()}")


In [ ]:
# @title 4) Utilities (pair scan, golden benchmark, split)
IMAGE_EXTS = {".png", ".jpg", ".jpeg"}

GROUP_IMAGE_DIRS: dict[str, tuple[str, ...]] = {
    "handwritten_original": ("original", "images", "handwritten"),
    "clean_original": ("original", "images", "clean"),
    "camera_distorted_original": ("original", "images", "camera"),
    "handwritten_corrupt": ("corrupt", "images", "handwritten"),
    "clean_corrupt": ("corrupt", "images", "clean"),
    "camera_distorted_corrupt": ("corrupt", "images", "camera"),
    "manual_handwritten": ("manual", "handwritten"),
    "manual_digital": ("manual", "digital"),
}

GROUP_ORDER = [
    "handwritten_original",
    "clean_original",
    "camera_distorted_original",
    "handwritten_corrupt",
    "clean_corrupt",
    "camera_distorted_corrupt",
    "manual_handwritten",
    "manual_digital",
]

GOLDEN_BENCHMARK_ENTRIES = [
    {"group": "handwritten_original", "filename": "000132946-1_21_1.jpg"},
    {"group": "handwritten_original", "filename": "230000659-1_3_1.jpg"},
    {"group": "handwritten_original", "filename": "220010817-1_18_1.jpg"},
    {"group": "handwritten_original", "filename": "220034811-1_2_1.jpg"},
    {"group": "handwritten_original", "filename": "190019918-1_1_1.jpg"},
    {"group": "handwritten_original", "filename": "212001709-1_4_1.jpg"},
    {"group": "handwritten_original", "filename": "150203639-1_2_1.jpg"},
    {"group": "handwritten_original", "filename": "220001519-1_1_1.jpg"},
    {"group": "handwritten_original", "filename": "000128225-1_1_1.jpg"},
    {"group": "handwritten_original", "filename": "000132422-1_1_1.jpg"},
    {"group": "clean_original", "filename": "000102395-1_1_1.png"},
    {"group": "clean_original", "filename": "000114963-1_1_1.png"},
    {"group": "clean_original", "filename": "110002051-1_1_1.png"},
    {"group": "clean_original", "filename": "220031110-1_1_1.png"},
    {"group": "clean_original", "filename": "000104407-1_1_1.png"},
    {"group": "clean_original", "filename": "190015389-1_1_1.png"},
    {"group": "clean_original", "filename": "000112762-1_1_1.png"},
    {"group": "clean_original", "filename": "000109599-1_2_1.png"},
    {"group": "clean_original", "filename": "212002605-1_1_1.png"},
    {"group": "clean_original", "filename": "000111721-1_1_1.png"},
    {"group": "camera_distorted_original", "filename": "000118903-1_1_1_distorted.jpg"},
    {"group": "camera_distorted_original", "filename": "000111686-1_1_1_distorted.jpg"},
    {"group": "camera_distorted_original", "filename": "210000061-1_4_1_distorted.jpg"},
    {"group": "camera_distorted_original", "filename": "230003675-1_30_1_distorted.jpg"},
    {"group": "camera_distorted_original", "filename": "000115397-1_1_1_distorted.jpg"},
    {"group": "camera_distorted_original", "filename": "220019566-1_1_1_distorted.jpg"},
    {"group": "camera_distorted_original", "filename": "210000094-1_7_1_distorted.jpg"},
    {"group": "camera_distorted_original", "filename": "190004306-1_1_1_distorted.jpg"},
    {"group": "camera_distorted_original", "filename": "000101628-1_1_1_distorted.jpg"},
    {"group": "camera_distorted_original", "filename": "211002852-1_1_1_distorted.jpg"},
    {"group": "handwritten_corrupt", "filename": "e3863145ef3e42afae97cd349163d0c6.jpg"},
    {"group": "handwritten_corrupt", "filename": "5e11951664a3457e9739d93a3171aa48.jpg"},
    {"group": "handwritten_corrupt", "filename": "86f558bff0ed4074ab55b765a9ba584d.jpg"},
    {"group": "handwritten_corrupt", "filename": "a8e9d04340e94054bb15e748c7927e1c.jpg"},
    {"group": "handwritten_corrupt", "filename": "4523f7f2dc624128bdcbc82bc118d335.jpg"},
    {"group": "handwritten_corrupt", "filename": "1bdd86927d97410db65cc7d1799735ae.jpg"},
    {"group": "handwritten_corrupt", "filename": "6717afa4e0144f71bd2c6300540be8d5.jpg"},
    {"group": "handwritten_corrupt", "filename": "dbd18f2c3b5144e9a937d85be0925c7a.jpg"},
    {"group": "handwritten_corrupt", "filename": "5537a1a3a4044c2997cf075c09e467fb.jpg"},
    {"group": "handwritten_corrupt", "filename": "6308c109bd69476893997da0bad95dc6.jpg"},
    {"group": "clean_corrupt", "filename": "aabc4e73b470487e92eec3a6f5f2dce8.png"},
    {"group": "clean_corrupt", "filename": "0ffd9e6dc1f742fabab24fafcfc72579.png"},
    {"group": "clean_corrupt", "filename": "95c0aa4542a8441790d08805168f182b.png"},
    {"group": "clean_corrupt", "filename": "beb2ea01c54c41d4bac1a9a7b424febf.png"},
    {"group": "clean_corrupt", "filename": "9e1d9aae9505446f8be8d8a92494c20d.png"},
    {"group": "clean_corrupt", "filename": "729021847bb641f0a7e840a64d993bf4.png"},
    {"group": "clean_corrupt", "filename": "9bc9e120643948ee8788e4295ecc22f1.png"},
    {"group": "clean_corrupt", "filename": "a0d9c58a64a74b4782cd135d353a33dc.png"},
    {"group": "clean_corrupt", "filename": "4541abd5c9d14406b27133995ce9ff23.png"},
    {"group": "clean_corrupt", "filename": "d2e5f36027ea4cb582f53c3eb3d3e59c.png"},
    {"group": "camera_distorted_corrupt", "filename": "d4eb58a8ff2640eab7a6a93bc4211a03.jpg"},
    {"group": "camera_distorted_corrupt", "filename": "ca4c0489b165467789f2146a2afeb74b.png"},
    {"group": "camera_distorted_corrupt", "filename": "37cbb51c3b2049b98db0c16fccb47e47.png"},
    {"group": "camera_distorted_corrupt", "filename": "5ea6e4a124894bb9839399772ed925f6.png"},
    {"group": "camera_distorted_corrupt", "filename": "982f0e40df14456795320bb9b26db89a.png"},
    {"group": "camera_distorted_corrupt", "filename": "e56c78c7987142b7a874540a8dab923d.png"},
    {"group": "camera_distorted_corrupt", "filename": "dcf91aef10f246a58325f20a889d7de7.png"},
    {"group": "camera_distorted_corrupt", "filename": "e6692210188a45ab8e7673d9719389d5.png"},
    {"group": "camera_distorted_corrupt", "filename": "4eb68ed427b24f0cb64c77f6656b1362.png"},
    {"group": "camera_distorted_corrupt", "filename": "2051043cf00b482aa76c122af3ab199b.png"},
    {"group": "manual_handwritten", "filename": "gs0.png"},
    {"group": "manual_handwritten", "filename": "gs1.png"},
    {"group": "manual_handwritten", "filename": "gs2.png"},
    {"group": "manual_handwritten", "filename": "gs3.png"},
    {"group": "manual_handwritten", "filename": "gs4.png"},
    {"group": "manual_handwritten", "filename": "gs5.png"},
    {"group": "manual_handwritten", "filename": "gs6.png"},
    {"group": "manual_handwritten", "filename": "gs7.png"},
    {"group": "manual_handwritten", "filename": "gs8.png"},
    {"group": "manual_handwritten", "filename": "gs9.png"},
    {"group": "manual_handwritten", "filename": "gs10.png"},
    {"group": "manual_handwritten", "filename": "gs11.png"},
    {"group": "manual_handwritten", "filename": "gs12.png"},
    {"group": "manual_handwritten", "filename": "gs13.png"},
    {"group": "manual_handwritten", "filename": "gs14.png"},
    {"group": "manual_handwritten", "filename": "gs15.png"},
    {"group": "manual_handwritten", "filename": "gs16.png"},
    {"group": "manual_handwritten", "filename": "gs17.png"},
    {"group": "manual_handwritten", "filename": "gs18.png"},
    {"group": "manual_handwritten", "filename": "gs19.png"},
    {"group": "manual_digital", "filename": "gs0.png"},
    {"group": "manual_digital", "filename": "gs1.png"},
    {"group": "manual_digital", "filename": "gs2.png"},
    {"group": "manual_digital", "filename": "gs3.png"},
    {"group": "manual_digital", "filename": "gs4.png"},
    {"group": "manual_digital", "filename": "gs5.png"},
    {"group": "manual_digital", "filename": "gs6.png"},
    {"group": "manual_digital", "filename": "gs7.png"},
    {"group": "manual_digital", "filename": "gs8.png"},
    {"group": "manual_digital", "filename": "gs9.png"},
    {"group": "manual_digital", "filename": "gs10.png"},
    {"group": "manual_digital", "filename": "gs11.png"},
    {"group": "manual_digital", "filename": "gs12.png"},
    {"group": "manual_digital", "filename": "gs13.png"},
    {"group": "manual_digital", "filename": "gs14.png"},
    {"group": "manual_digital", "filename": "gs15.png"},
    {"group": "manual_digital", "filename": "gs16.png"},
    {"group": "manual_digital", "filename": "gs17.png"},
    {"group": "manual_digital", "filename": "gs18.png"},
    {"group": "manual_digital", "filename": "gs19.png"},
]


def normalize_semantic(text: str) -> str:
    return re.sub(r"\s+", " ", text.strip())


def incipit_id_from_filename(filename: str) -> str:
    stem = Path(filename).stem
    if stem.endswith("_distorted"):
        stem = stem[: -len("_distorted")]
    return stem


def exclusion_id_for_entry(group: str, filename: str) -> str:
    if group.startswith("manual_") or group.endswith("_corrupt"):
        return Path(filename).stem
    return incipit_id_from_filename(filename)


def golden_image_path(golden_dir: Path, group: str, filename: str) -> Path:
    return golden_dir.joinpath(*GROUP_IMAGE_DIRS[group]) / filename


def golden_gt_path(golden_dir: Path, group: str, filename: str) -> Path:
    stem = Path(filename).stem
    if group.startswith("manual_"):
        return golden_dir / "manual" / "ground_truth" / f"{stem}.semantic"
    if group.endswith("_corrupt"):
        return golden_dir / "corrupt" / "ground_truth" / f"{stem}.semantic"
    return golden_dir / "original" / "ground_truth" / f"{incipit_id_from_filename(filename)}.semantic"


def load_golden_selection(golden_dir: Path) -> list[dict]:
    """Prefer Drive benchmark JSONs; fall back to inline GOLDEN_BENCHMARK_ENTRIES."""
    bench_dir = golden_dir / "benchmark"
    sel_files = [
        "selection_10.json",
        "selection_10_corrupt.json",
        "selection_manual.json",
    ]
    if bench_dir.is_dir() and all((bench_dir / name).is_file() for name in sel_files):
        selection: list[dict] = []
        for name in sel_files:
            selection.extend(json.loads((bench_dir / name).read_text(encoding="utf-8")))
        print(f"Loaded golden benchmark from {bench_dir} ({len(selection)} samples)")
        return selection

    print(f"Using inline golden benchmark list ({len(GOLDEN_BENCHMARK_ENTRIES)} samples)")
    return list(GOLDEN_BENCHMARK_ENTRIES)


def build_golden_eval_records(golden_dir: Path, selection: list[dict]) -> tuple[list[dict], set[str]]:
    records: list[dict] = []
    exclusion_ids: set[str] = set()
    missing = 0

    for entry in selection:
        group = entry["group"]
        filename = entry["filename"]
        exclusion_ids.add(exclusion_id_for_entry(group, filename))

        img_path = golden_image_path(golden_dir, group, filename)
        gt_path = golden_gt_path(golden_dir, group, filename)
        if not img_path.is_file() or not gt_path.is_file():
            missing += 1
            continue

        text = normalize_semantic(gt_path.read_text(encoding="utf-8"))
        records.append(
            {
                "group": group,
                "filename": filename,
                "stem": img_path.stem,
                "image": str(img_path),
                "gt": str(gt_path),
                "text": text,
                "exclusion_id": exclusion_id_for_entry(group, filename),
            }
        )

    if missing:
        print(
            f"Warning: {missing} golden benchmark files missing under {golden_dir}. "
            "Upload the full golden_set/ folder for eval."
        )
    return records, exclusion_ids


STYLE_GROUP_PREFIXES = {
    "handwritten": frozenset({"mashcima", "corrupt_handwritten"}),
    "clean": frozenset({"primus", "corrupt_clean", "corrupt_beamed_clean"}),
    "camera": frozenset({"camera_primus", "corrupt_camera", "corrupt_beamed_camera"}),
}


def record_style(group_prefix: str) -> str:
    for style, prefixes in STYLE_GROUP_PREFIXES.items():
        if group_prefix in prefixes:
            return style
    if "handwritten" in group_prefix:
        return "handwritten"
    if "camera" in group_prefix:
        return "camera"
    if group_prefix.endswith("_clean") or group_prefix == "primus":
        return "clean"
    raise ValueError(f"Unknown style for group_prefix={group_prefix!r}")


def record_is_corrupt(group_prefix: str) -> bool:
    return group_prefix.startswith("corrupt")


def style_corrupt_quotas(
    max_pairs: int,
    style_weights: dict[str, float],
    corrupt_fraction: float,
) -> dict[tuple[str, bool], int]:
    """Target counts per (style, is_corrupt) for stratified subsampling."""
    quotas: dict[tuple[str, bool], int] = {}
    assigned = 0
    active = [(s, w) for s, w in style_weights.items() if w > 0]
    for i, (style, weight) in enumerate(active):
        if i == len(active) - 1:
            style_total = max(0, max_pairs - assigned)
        else:
            style_total = int(round(max_pairs * weight))
            assigned += style_total
        corrupt_n = int(round(style_total * corrupt_fraction))
        orig_n = style_total - corrupt_n
        quotas[(style, False)] = orig_n
        quotas[(style, True)] = corrupt_n
    return quotas


def build_style_sample_weights(
    records: list[dict],
    style_weights: dict[str, float],
    corrupt_fraction: float = 0.5,
):
    import torch
    from collections import defaultdict

    buckets: dict[tuple[str, bool], list[int]] = defaultdict(list)
    for i, rec in enumerate(records):
        gp = rec.get("group_prefix") or rec["stem"].split("__", 1)[0]
        buckets[(record_style(gp), record_is_corrupt(gp))].append(i)

    index_weights = [0.0] * len(records)
    for style, target_share in style_weights.items():
        if target_share <= 0:
            continue
        branches = []
        for corrupt in (False, True):
            idxs = buckets.get((style, corrupt), [])
            if idxs:
                branches.append(idxs)
        if not branches:
            continue
        per_branch = target_share / len(branches)
        for idxs in branches:
            w = per_branch / len(idxs)
            for i in idxs:
                index_weights[i] = w

    weights = torch.tensor(index_weights, dtype=torch.double)
    if weights.sum() <= 0:
        return torch.ones(len(records), dtype=torch.double)
    return weights / weights.sum() * len(records)


def print_style_mix_stats(records: list[dict], weights=None) -> None:
    from collections import Counter

    def bucket(rec):
        gp = rec.get("group_prefix") or rec["stem"].split("__", 1)[0]
        return (record_style(gp), record_is_corrupt(gp))

    raw = Counter(bucket(r) for r in records)
    print("Train pool by (style, corrupt):")
    for key in sorted(raw.keys()):
        print(f"  {key[0]:12s} corrupt={key[1]}: {raw[key]:,}")

    if weights is not None:
        weighted = Counter()
        for rec, wt in zip(records, weights.tolist()):
            weighted[bucket(rec)] += wt
        total = sum(weighted.values())
        print("Effective sampler mix (weights normalized):")
        for key in sorted(weighted.keys()):
            pct = 100.0 * weighted[key] / total if total else 0
            print(f"  {key[0]:12s} corrupt={key[1]}: {pct:.1f}%")


def build_pairs(images_dir: Path, gt_dir: Path) -> list[dict]:
    """Pair consolidated images with same-stem .semantic files."""
    if not images_dir.is_dir() or not gt_dir.is_dir():
        raise FileNotFoundError(
            f"Missing data dirs. Expected:\n  {images_dir}\n  {gt_dir}\n"
            "Upload data/consolidated/ to Drive first (see README_colab.md)."
        )

    pairs: list[dict] = []
    missing_gt = 0
    for img_path in sorted(images_dir.iterdir()):
        if not img_path.is_file() or img_path.suffix.lower() not in IMAGE_EXTS:
            continue
        gt_path = gt_dir / f"{img_path.stem}.semantic"
        if not gt_path.is_file():
            missing_gt += 1
            continue
        text = normalize_semantic(gt_path.read_text(encoding="utf-8"))
        if not text:
            continue
        match_stem = img_path.stem.split("__", 1)[-1] if "__" in img_path.stem else img_path.stem
        pairs.append(
            {
                "stem": img_path.stem,
                "match_stem": match_stem,
                "image": str(img_path),
                "gt": str(gt_path),
                "text": text,
                "group_prefix": img_path.stem.split("__", 1)[0],
            }
        )

    print(f"Built {len(pairs):,} pairs ({missing_gt:,} images missing GT)")
    return pairs


def pair_is_leaked(pair: dict, exclusion_ids: set[str]) -> bool:
    if pair["stem"].startswith("golden_"):
        return True
    return pair.get("match_stem", pair["stem"]) in exclusion_ids


def split_pairs(pairs: list[dict], exclusion_ids: set[str]) -> dict[str, list[dict]]:
    leaked = [p for p in pairs if pair_is_leaked(p, exclusion_ids)]
    train = [p for p in pairs if not pair_is_leaked(p, exclusion_ids)]
    return {"train": train, "leaked_from_consolidated": leaked}


def save_manifest(manifest: dict) -> None:
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print(f"Wrote {MANIFEST_PATH}")


def load_or_build_manifest() -> dict:
    if MANIFEST_PATH.is_file():
        cached = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
        if cached.get("version") == MANIFEST_VERSION:
            print(f"Loaded cached manifest from {MANIFEST_PATH} (version {MANIFEST_VERSION})")
            print(
                f"train={len(cached['splits']['train']):,}  "
                f"eval_golden={len(cached['splits']['eval_golden']):,}"
            )
            return cached
        print(f"Cached manifest version {cached.get('version')!r} != {MANIFEST_VERSION} -- rebuilding")

    selection = load_golden_selection(GOLDEN_DIR)
    eval_golden, exclusion_ids = build_golden_eval_records(GOLDEN_DIR, selection)
    pairs = build_pairs(IMAGES_DIR, GT_DIR)
    splits = split_pairs(pairs, exclusion_ids)

    manifest = {
        "version": MANIFEST_VERSION,
        "images_dir": str(IMAGES_DIR),
        "gt_dir": str(GT_DIR),
        "golden_dir": str(GOLDEN_DIR),
        "train_pairs": len(splits["train"]),
        "eval_golden": len(eval_golden),
        "splits": {"train": splits["train"], "eval_golden": eval_golden},
    }
    save_manifest(manifest)
    print(
        f"train={len(splits['train']):,}  eval_golden={len(eval_golden):,}  "
        f"leaked_excluded={len(splits['leaked_from_consolidated']):,}"
    )
    return manifest


def normalize_tokens(text: str) -> list[str]:
    return [p for p in re.split(r"[\s\t]+", text.strip()) if p]


# Relative major/minor share the same accidentals; canonicalize to the major form.
_RELATIVE_MINOR_TO_MAJOR = {
    "Am": "CM",
    "Em": "GM",
    "Bm": "DM",
    "F#m": "AM",
    "C#m": "EM",
    "G#m": "BM",
    "D#m": "F#M",
    "A#m": "C#M",
    "Dm": "FM",
    "Gm": "BbM",
    "Cm": "EbM",
    "Fm": "AbM",
    "Bbm": "DbM",
    "Ebm": "GbM",
    "Abm": "CbM",
}

# Common time ≡ 4/4; cut time ≡ 2/2.
_TIME_SIGNATURE_CANON = {
    "timeSignature-4/4": "timeSignature-C",
    "timeSignature-C": "timeSignature-C",
    "timeSignature-2/2": "timeSignature-C/",
    "timeSignature-C/": "timeSignature-C/",
}


def canonicalize_token(token: str) -> str:
    """Map notation-equivalent tokens to a shared canonical form for scoring."""
    if token.startswith("keySignature-"):
        key = token[len("keySignature-") :]
        key = _RELATIVE_MINOR_TO_MAJOR.get(key, key)
        return f"keySignature-{key}"
    return _TIME_SIGNATURE_CANON.get(token, token)


def canonicalize_tokens(tokens: list[str]) -> list[str]:
    return [canonicalize_token(t) for t in tokens]


def token_edit_distance(pred: list[str], gt: list[str]) -> int:
    """Token-level Levenshtein (same metric as scripts/omr_benchmark_score.py).

    Relative major/minor key signatures and common/cut-time aliases count as equal.
    """
    pred = canonicalize_tokens(pred)
    gt = canonicalize_tokens(gt)
    m, n = len(gt), len(pred)
    if m == 0:
        return n
    if n == 0:
        return m
    prev = list(range(n + 1))
    for i in range(1, m + 1):
        curr = [i] + [0] * n
        gt_tok = gt[i - 1]
        for j in range(1, n + 1):
            if gt_tok == pred[j - 1]:
                curr[j] = prev[j - 1]
            else:
                curr[j] = 1 + min(prev[j], curr[j - 1], prev[j - 1])
        prev = curr
    return prev[n]


def strip_trailing_barline(tokens: list[str]) -> list[str]:
    """Drop a single trailing 'barline' token if present (trailing-barline-agnostic eval)."""
    if tokens and tokens[-1] == "barline":
        return tokens[:-1]
    return tokens


def barline_forgiven_distance(pred: list[str], gt: list[str]) -> int:
    """Token edit distance that ignores a trailing-barline-only difference and never penalizes.

    The min over {plain, strip-pred, strip-gt, strip-both} is always <= the plain distance,
    so forgiving a trailing barline can only help or be neutral, while any difference that is
    solely a trailing barline (on either side) is neutralized. Only one trailing 'barline' per
    side is ever removed; internal barlines stay counted.
    """
    sp, sg = strip_trailing_barline(pred), strip_trailing_barline(gt)
    return min(
        token_edit_distance(pred, gt),
        token_edit_distance(sp, gt),
        token_edit_distance(pred, sg),
        token_edit_distance(sp, sg),
    )


def score_tokens(pred: list[str], gt: list[str]) -> dict:
    n_gt = len(gt)
    errors = barline_forgiven_distance(pred, gt)
    if n_gt == 0:
        return {
            "errors": errors,
            "gt_tokens": 0,
            "error_rate_pct": 100.0 if errors else 0.0,
            "accuracy_pct": 0.0 if errors else 100.0,
        }
    error_rate_pct = errors / n_gt * 100.0
    accuracy_pct = max(0.0, 100.0 - error_rate_pct)
    return {
        "errors": errors,
        "gt_tokens": n_gt,
        "error_rate_pct": round(error_rate_pct, 2),
        "accuracy_pct": round(accuracy_pct, 2),
    }


def group_sort_key(group: str) -> int:
    try:
        return GROUP_ORDER.index(group)
    except ValueError:
        return len(GROUP_ORDER)


def predict_semantic(
    model,
    processor,
    image_path: str,
    *,
    device: str,
    task_prompt: str,
    max_new_tokens: int = 512,
) -> str:
    from PIL import Image

    image = Image.open(image_path).convert("RGB")
    inputs = processor(text=task_prompt, images=image, return_tensors="pt")
    model_dtype = next(model.parameters()).dtype
    cast_inputs = {}
    for k, v in inputs.items():
        if not hasattr(v, "to"):
            cast_inputs[k] = v
            continue
        v = v.to(device)
        if k == "pixel_values":
            v = v.to(dtype=model_dtype)
        cast_inputs[k] = v
    inputs = cast_inputs

    gen_kwargs = dict(
        input_ids=inputs["input_ids"],
        pixel_values=inputs["pixel_values"],
        max_new_tokens=max_new_tokens,
        num_beams=1,  # beam search + constrained processor collapses to zero tokens; greedy is equivalent here
        do_sample=False,
        use_cache=False,
    )
    if USE_CONSTRAINED_DECODE:
        gen_kwargs["logits_processor"] = [make_primus_logits_processor(processor.tokenizer)]

    model.eval()
    with torch.inference_mode():
        generated_ids = model.generate(**gen_kwargs)

    # Fine-tuned PrIMuS semantics are plain token strings -- do NOT run the native
    # <OCR> post-processor (it strips spaces and parses bbox-tagged OCR regions).
    prompt_ids = inputs["input_ids"][0]
    gen_ids = generated_ids[0]
    if (
        gen_ids.shape[0] >= prompt_ids.shape[0]
        and torch.equal(gen_ids[: prompt_ids.shape[0]], prompt_ids)
    ):
        new_ids = gen_ids[prompt_ids.shape[0] :]
    else:
        new_ids = gen_ids
    text = processor.tokenizer.decode(new_ids, skip_special_tokens=True)
    if text.startswith(task_prompt):
        text = text[len(task_prompt):]
    return normalize_semantic(text)


def summarize_golden_scores(sample_rows: list[dict]) -> dict:
    from collections import defaultdict

    by_group: dict[str, list[dict]] = defaultdict(list)
    for row in sample_rows:
        by_group[row["group"]].append(row)

    group_summary = {}
    for group in sorted(by_group, key=group_sort_key):
        rows = by_group[group]
        group_summary[group] = {
            "n": len(rows),
            "exact_match": sum(1 for r in rows if r["errors"] == 0),
            "mean_error_rate_pct": round(sum(r["error_rate_pct"] for r in rows) / len(rows), 2),
            "mean_accuracy_pct": round(sum(r["accuracy_pct"] for r in rows) / len(rows), 2),
        }

    return {
        "exact_match": sum(1 for r in sample_rows if r["errors"] == 0),
        "mean_error_rate_pct": round(
            sum(r["error_rate_pct"] for r in sample_rows) / max(len(sample_rows), 1), 2
        ),
        "mean_accuracy_pct": round(
            sum(r["accuracy_pct"] for r in sample_rows) / max(len(sample_rows), 1), 2
        ),
    }, group_summary


def run_golden_eval(
    model,
    processor,
    golden_records: list[dict],
    *,
    phase: str,
    model_id: str,
    task_prompt: str,
    device: str,
    use_lora: bool = False,
    show_progress: bool = True,
) -> dict:
    from tqdm.auto import tqdm

    if not golden_records:
        print(
            "No golden eval samples found. Upload golden_set/ to Drive "
            f"(expected under {GOLDEN_DIR})."
        )
        return {
            "phase": phase,
            "model_id": model_id,
            "task_prompt": task_prompt,
            "use_lora": use_lora,
            "tokenizer": "primus_closed_vocab",
            "barline_normalization": "trailing_barline_ignored",
            "vocab_size": processor.tokenizer.vocab_size,
            "n_samples": 0,
            "expected_samples": GOLDEN_BENCHMARK_COUNT,
            "overall": {},
            "by_group": {},
            "samples": [],
        }

    model.to(device)
    sample_rows = []
    iterator = tqdm(golden_records, desc=f"golden eval ({phase})") if show_progress else golden_records

    for rec in iterator:
        gt_tokens = normalize_tokens(rec["text"])
        pred_text = predict_semantic(
            model,
            processor,
            rec["image"],
            device=device,
            task_prompt=task_prompt,
        )
        pred_tokens = normalize_tokens(pred_text)
        scored = score_tokens(pred_tokens, gt_tokens)
        sample_rows.append(
            {
                "group": rec["group"],
                "filename": rec["filename"],
                "gt_tokens": scored["gt_tokens"],
                "pred_tokens": len(pred_tokens),
                "errors": scored["errors"],
                "error_rate_pct": scored["error_rate_pct"],
                "accuracy_pct": scored["accuracy_pct"],
                "prediction": pred_text,
            }
        )

    overall, by_group = summarize_golden_scores(sample_rows)
    return {
        "phase": phase,
        "model_id": model_id,
        "task_prompt": task_prompt,
        "use_lora": use_lora,
        "tokenizer": "primus_closed_vocab",
        "barline_normalization": "trailing_barline_ignored",
        "vocab_size": processor.tokenizer.vocab_size,
        "n_samples": len(sample_rows),
        "expected_samples": GOLDEN_BENCHMARK_COUNT,
        "overall": overall,
        "by_group": by_group,
        "samples": sample_rows,
    }


def print_golden_eval_summary(results: dict, title: str) -> None:
    if not results.get("samples"):
        print(f"{title}: no samples scored")
        return

    n = results["n_samples"]
    overall = results["overall"]
    print(f"{title}: {n}/{results['expected_samples']} samples")
    print(
        f"Exact match: {overall['exact_match']}/{n} "
        f"({overall['exact_match'] / n * 100:.1f}%)"
    )
    print(f"Mean token accuracy: {overall['mean_accuracy_pct']:.2f}%")
    for group in sorted(results["by_group"], key=group_sort_key):
        g = results["by_group"][group]
        print(
            f"  {group}: n={g['n']}  acc={g['mean_accuracy_pct']:.2f}%  "
            f"err={g['mean_error_rate_pct']:.2f}%"
        )


def compare_golden_eval_results(before: dict, after: dict) -> None:
    if not before.get("samples") or not after.get("samples"):
        print("Cannot compare -- missing before or after results")
        return

    b_overall = before["overall"]["mean_accuracy_pct"]
    a_overall = after["overall"]["mean_accuracy_pct"]
    print("\n=== Before vs after (golden 100) ===")
    print(f"Overall accuracy: {b_overall:.2f}% -> {a_overall:.2f}%  (delta {a_overall - b_overall:+.2f} pp)")

    groups = sorted(
        set(before["by_group"]) | set(after["by_group"]),
        key=group_sort_key,
    )
    for group in groups:
        b_acc = before["by_group"].get(group, {}).get("mean_accuracy_pct")
        a_acc = after["by_group"].get(group, {}).get("mean_accuracy_pct")
        if b_acc is None or a_acc is None:
            continue
        print(f"  {group}: {b_acc:.2f}% -> {a_acc:.2f}%  (delta {a_acc - b_acc:+.2f} pp)")



def golden_eval_epoch_path(epoch: int) -> Path:
    return EVAL_GOLDEN_EPOCH_DIR / f"eval_golden_epoch_{epoch:02d}.json"


def format_golden_eval_writeup_section(
    results: dict,
    *,
    epoch: int,
    baseline: dict | None = None,
    previous: dict | None = None,
) -> str:
    lines: list[str] = []
    overall = results.get("overall") or {}
    n = results.get("n_samples", 0)
    mean_acc = overall.get("mean_accuracy_pct")
    exact = overall.get("exact_match")
    lines.append(f"## Epoch {epoch}")
    lines.append("")
    if mean_acc is None:
        lines.append("No samples scored.")
        lines.append("")
        return "\n".join(lines)

    lines.append(
        f"- **Mean token accuracy:** {mean_acc:.2f}%  "
        f"({exact}/{n} exact match)"
    )
    if baseline and baseline.get("overall"):
        b = baseline["overall"]["mean_accuracy_pct"]
        lines.append(f"- **vs baseline:** {mean_acc - b:+.2f} pp")
    if previous and previous.get("overall"):
        p = previous["overall"]["mean_accuracy_pct"]
        lines.append(f"- **vs previous epoch:** {mean_acc - p:+.2f} pp")
    lines.append("")
    lines.append("| Group | n | Accuracy | Error rate |")
    lines.append("|---|---:|---:|---:|")
    for group in sorted(results.get("by_group") or {}, key=group_sort_key):
        g = results["by_group"][group]
        lines.append(
            f"| {group} | {g['n']} | {g['mean_accuracy_pct']:.2f}% | "
            f"{g['mean_error_rate_pct']:.2f}% |"
        )
    lines.append("")
    return "\n".join(lines)


def save_golden_epoch_eval(
    results: dict,
    *,
    epoch: int,
    baseline: dict | None = None,
    history: list[dict] | None = None,
) -> dict:
    """Persist per-epoch golden JSON + rolling markdown writeup + history index."""
    EVAL_GOLDEN_EPOCH_DIR.mkdir(parents=True, exist_ok=True)
    path = golden_eval_epoch_path(epoch)
    path.write_text(json.dumps(results, indent=2), encoding="utf-8")

    if history is None:
        if EVAL_GOLDEN_EPOCH_HISTORY_PATH.is_file():
            history = json.loads(
                EVAL_GOLDEN_EPOCH_HISTORY_PATH.read_text(encoding="utf-8")
            )
        else:
            history = []

    history = [h for h in history if h.get("epoch") != epoch]
    summary = {
        "epoch": epoch,
        "phase": results.get("phase"),
        "path": str(path),
        "n_samples": results.get("n_samples"),
        "mean_accuracy_pct": (results.get("overall") or {}).get("mean_accuracy_pct"),
        "exact_match": (results.get("overall") or {}).get("exact_match"),
        "by_group": {
            g: {"mean_accuracy_pct": v["mean_accuracy_pct"], "n": v["n"]}
            for g, v in (results.get("by_group") or {}).items()
        },
    }
    history.append(summary)
    history.sort(key=lambda h: h["epoch"])
    EVAL_GOLDEN_EPOCH_HISTORY_PATH.write_text(
        json.dumps(history, indent=2), encoding="utf-8"
    )

    sections = [
        "# Golden benchmark — per-epoch results",
        "",
        f"Output dir: `{DRIVE_OUTPUT_DIR}`",
        "",
    ]
    if baseline and baseline.get("overall"):
        b = baseline["overall"]
        sections.append(
            f"**Baseline (before training):** {b['mean_accuracy_pct']:.2f}% "
            f"mean token accuracy "
            f"({b.get('exact_match', '?')}/{baseline.get('n_samples', '?')} exact)."
        )
        sections.append("")

    previous = None
    for entry in history:
        ep = int(entry["epoch"])
        ep_path = Path(entry["path"]) if entry.get("path") else golden_eval_epoch_path(ep)
        if not ep_path.is_file():
            ep_path = golden_eval_epoch_path(ep)
        if ep_path.is_file():
            ep_results = json.loads(ep_path.read_text(encoding="utf-8"))
        else:
            ep_results = {
                "overall": {
                    "mean_accuracy_pct": entry.get("mean_accuracy_pct"),
                    "exact_match": entry.get("exact_match"),
                },
                "n_samples": entry.get("n_samples"),
                "by_group": {},
            }
        sections.append(
            format_golden_eval_writeup_section(
                ep_results,
                epoch=ep,
                baseline=baseline,
                previous=previous,
            )
        )
        previous = ep_results

    EVAL_GOLDEN_EPOCH_WRITEUP_PATH.write_text(
        "\n".join(sections).rstrip() + "\n", encoding="utf-8"
    )
    print(f"Wrote {path}")
    print(f"Wrote {EVAL_GOLDEN_EPOCH_HISTORY_PATH}")
    print(f"Wrote {EVAL_GOLDEN_EPOCH_WRITEUP_PATH}")
    return summary


manifest = load_or_build_manifest()


In [ ]:
# @title 5) Dataset + collator
from PIL import Image
from torch.utils.data import Dataset

# If any source image is truncated/corrupt, do not feed half-decoded pixels with a
# valid label into training. __getitem__ treats an undecodable file as a miss and
# substitutes a valid sample while reporting the bad path once.

_CORRUPT_WARNED: set[str] = set()


class OMRDataset(Dataset):
    def __init__(self, records: list[dict]):
        self.records = records

    def __len__(self) -> int:
        return len(self.records)

    @staticmethod
    def _load_rgb(path: str) -> "Image.Image":
        image = Image.open(path).convert("RGB")
        image.load()  # force full decode now so any error surfaces here, not mid-batch
        return image

    def __getitem__(self, idx: int) -> dict:
        rec = self.records[idx]
        try:
            image = self._load_rgb(rec["image"])
        except (OSError, SyntaxError, ValueError) as exc:
            # Warn once, then substitute the next valid sample so the batch stays intact.
            # Run a source-data audit to remove any reported files permanently.
            if rec["image"] not in _CORRUPT_WARNED:
                _CORRUPT_WARNED.add(rec["image"])
                print(f"[OMRDataset] skipping unreadable image: {rec['image']} ({exc})", flush=True)
            n = len(self.records)
            for off in range(1, n):
                alt = self.records[(idx + off) % n]
                try:
                    image = self._load_rgb(alt["image"])
                    rec = alt
                    break
                except (OSError, SyntaxError, ValueError):
                    continue
            else:
                raise RuntimeError("No decodable images in dataset") from exc
        return {"image": image, "text": rec["text"], "stem": rec["stem"]}


def make_collate_fn(processor, task_prompt: str, max_label_length: int):
    """Florence-2 is an ENCODER-DECODER model.

    The encoder consumes (task prompt + image); the decoder target is the answer.
    Do NOT concatenate prompt+answer into input_ids (that is the causal-LM recipe
    and makes the model echo an answer it can already see in the encoder input --
    at inference the encoder only sees the prompt, so it emits EOS => empty output).

    Pass the answer as `labels`; the model builds decoder_input_ids internally via
    shift_tokens_right(labels, ...). See modeling_florence2.py forward().
    """

    def collate(batch: list[dict]) -> dict:
        images = [b["image"] for b in batch]
        texts = [b["text"] for b in batch]

        # Encoder inputs: task prompt + image.
        enc = processor(
            text=[task_prompt] * len(batch),
            images=images,
            return_tensors="pt",
            padding=True,
        )

        # Decoder targets: one token ID per PrIMuS semantic type.
        labels = encode_semantic_labels(
            processor.tokenizer,
            texts,
            max_length=max_label_length,
        )

        # Ignore pad positions in the loss.
        pad_id = processor.tokenizer.pad_token_id
        if pad_id is not None:
            labels[labels == pad_id] = -100

        enc["labels"] = labels
        return enc

    return collate


train_ds = OMRDataset(manifest["splits"]["train"])
golden_ds = OMRDataset(manifest["splits"]["eval_golden"])

train_sample_weights = build_style_sample_weights(
    manifest["splits"]["train"],
    STYLE_WEIGHTS,
    STYLE_WEIGHT_CLEAN_VS_CORRUPT,
)
print_style_mix_stats(manifest["splits"]["train"], train_sample_weights)

print(f"train={len(train_ds):,}  golden_eval={len(golden_ds):,}")

In [ ]:
# @title 6) Load Florence-2-large model + processor (A100 bf16)
# Florence-2 remote code does not currently dispatch through Transformers SDPA;
# explicitly use eager attention to avoid _sdpa_can_dispatch model-load failures.
# LoRA adapters are applied in the next cell after baseline eval.
# Full compat shim inlined here so model load works even if utilities cell was skipped.

if "MODEL_ID" not in globals() or not MODEL_ID:
    raise RuntimeError(
        "MODEL_ID is not set â€” run the configuration cell (section 3) first."
    )
print(f"MODEL_ID={MODEL_ID!r}")


def patch_florence2_transformers5_compat(*, model_id: str | None = None) -> None:
    """Florence-2 shims for transformers 5.x (scripts/florence2_transformers5_compat.py)."""
    import transformers.configuration_utils as config_utils
    import transformers.tokenization_utils_base as tok_base

    _flag = "_florence2_tf5_compat_patched"

    if not getattr(config_utils.PretrainedConfig, _flag, False):
        _orig_init = config_utils.PretrainedConfig.__init__

        def _patched_init(self, *args, **kwargs):
            _orig_init(self, *args, **kwargs)
            if not hasattr(self, "forced_bos_token_id"):
                self.forced_bos_token_id = kwargs.get("forced_bos_token_id", None)

        config_utils.PretrainedConfig.__init__ = _patched_init
        setattr(config_utils.PretrainedConfig, _flag, True)

    ga_flag = "_florence2_tf5_getattr_patched"
    if not getattr(config_utils.PretrainedConfig, ga_flag, False):
        _orig_ga = config_utils.PretrainedConfig.__getattribute__

        def _patched_ga(self, key):
            if key == "forced_bos_token_id":
                try:
                    return _orig_ga(self, key)
                except AttributeError:
                    return None
            return _orig_ga(self, key)

        config_utils.PretrainedConfig.__getattribute__ = _patched_ga
        setattr(config_utils.PretrainedConfig, ga_flag, True)

    tok_flag = "_florence2_tf5_tok_getattr_patched"
    if not getattr(tok_base.PreTrainedTokenizerBase, tok_flag, False):
        if not hasattr(tok_base.PreTrainedTokenizerBase, "__getattr__"):
            return
        _orig_getattr = tok_base.PreTrainedTokenizerBase.__getattr__

        def _patched_getattr(self, name):
            if name == "additional_special_tokens":
                try:
                    return _orig_getattr(self, name)
                except AttributeError:
                    try:
                        added = _orig_getattr(self, "added_tokens_decoder")
                    except AttributeError:
                        return []
                    return [
                        tok.content
                        for tok in added.values()
                        if getattr(tok, "special", False)
                    ]
            return _orig_getattr(self, name)

        tok_base.PreTrainedTokenizerBase.__getattr__ = _patched_getattr
        setattr(tok_base.PreTrainedTokenizerBase, tok_flag, True)

    if model_id:
        try:
            from huggingface_hub import snapshot_download

            snapshot_download(
                model_id,
                allow_patterns=[
                    "configuration_florence2.py",
                    "processing_florence2.py",
                    "modeling_florence2.py",
                ],
            )
        except Exception:
            pass

    search_roots = [
        Path.home() / ".cache" / "huggingface" / "modules" / "transformers_modules",
        Path.home() / ".cache" / "huggingface" / "hub",
    ]
    patches = [
        (
            "configuration_florence2.py",
            "if self.forced_bos_token_id is None and kwargs.get(",
            "if getattr(self, 'forced_bos_token_id', None) is None and kwargs.get(",
        ),
        (
            "processing_florence2.py",
            "tokenizer.additional_special_tokens + \\",
            "getattr(tokenizer, 'additional_special_tokens', []) + \\",
        ),
        (
            "modeling_florence2.py",
            "        return self.language_model._supports_sdpa",
            "        if not hasattr(self, \"language_model\"):\n            return False\n        return self.language_model._supports_sdpa",
        ),
        (
            "modeling_florence2.py",
            "        return self.language_model._supports_flash_attn_2",
            "        if not hasattr(self, \"language_model\"):\n            return False\n        return self.language_model._supports_flash_attn_2",
        ),
    ]

    patched_any = False
    for root in search_roots:
        if not root.is_dir():
            continue
        for src in root.rglob("*.py"):
            for filename, old, new in patches:
                if src.name != filename:
                    continue
                try:
                    text = src.read_text(encoding="utf-8")
                except OSError:
                    continue
                if old in text and new not in text:
                    src.write_text(text.replace(old, new), encoding="utf-8")
                    patched_any = True

    modules_cache = search_roots[0]
    if patched_any and modules_cache.is_dir():
        for d in modules_cache.glob("*lorence*"):
            if d.is_dir():
                shutil.rmtree(d, ignore_errors=True)


patch_florence2_transformers5_compat(model_id=MODEL_ID)
print(f"Florence-2 compat applied with model_id={MODEL_ID!r}")

from transformers import AutoModelForCausalLM

processor = build_florence2_primus_processor(
    MODEL_ID,
    vocab_path=PRIMUS_VOCAB_PATH,
    trust_remote_code=True,
)
print(f"PrIMuS tokenizer vocab_size={processor.tokenizer.vocab_size:,}")

dtype = torch.bfloat16
print(f"Loading {MODEL_ID} ({dtype}, eager attention)...")

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=dtype,
    attn_implementation="eager",
    low_cpu_mem_usage=True,
)
resize_model_embeddings_for_primus(base_model, processor.tokenizer)
base_model.config.use_cache = False
print(
    "Base model loaded with resized PrIMuS embeddings "
    "(random-init decoder rows â€” retrain required)"
)

## Baseline eval (before training)

Zero-shot inference on all 100 golden benchmark images using the pretrained base model (no LoRA). Results are saved to `eval_golden_before.json` on Drive.

Set `SKIP_BASELINE_EVAL = True` in config to skip (e.g. when resuming a long run where baseline was already scored).

In [ ]:
# @title 6b) Baseline golden benchmark (zero-shot, before training)
golden_records = manifest["splits"]["eval_golden"]

if SKIP_BASELINE_EVAL and EVAL_GOLDEN_BEFORE_PATH.is_file():
    baseline_results = json.loads(EVAL_GOLDEN_BEFORE_PATH.read_text(encoding="utf-8"))
    print(f"SKIP_BASELINE_EVAL=True â€” loaded cached baseline from {EVAL_GOLDEN_BEFORE_PATH}")
    print_golden_eval_summary(baseline_results, "Baseline (cached)")
elif SKIP_BASELINE_EVAL:
    baseline_results = None
    print("SKIP_BASELINE_EVAL=True and no eval_golden_before.json found â€” skipping baseline")
else:
    baseline_results = run_golden_eval(
        base_model,
        processor,
        golden_records,
        phase="before_training",
        model_id=MODEL_ID,
        task_prompt=TASK_PROMPT,
        device=DEVICE,
        use_lora=False,
    )
    print_golden_eval_summary(baseline_results, "Baseline (zero-shot)")
    EVAL_GOLDEN_BEFORE_PATH.write_text(json.dumps(baseline_results, indent=2), encoding="utf-8")
    print(f"Wrote {EVAL_GOLDEN_BEFORE_PATH}")

In [ ]:
# @title 6c) Apply LoRA adapters (vision + language)
# A100 runs Florence-2-large in native bf16; 4-bit QLoRA is intentionally avoided to
# preserve fine-grained symbol discrimination and eliminate dequantization overhead.
from peft import LoraConfig, get_peft_model
from transformers import Trainer, TrainingArguments, TrainerCallback

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="SEQ_2_SEQ_LM",  # Florence-2 is encoder-decoder, NOT causal LM
    target_modules=[
        # Language attention. NOTE: the previous "o_proj" matched NOTHING -- Florence-2's
        # attention output projection is named "out_proj" (modeling_florence2.py). So the
        # earlier fine-tune never adapted the attention output at all; this fixes it.
        "q_proj",
        "k_proj",
        "v_proj",
        "out_proj",
        # Encoder+decoder FFN -- widen adapter capacity (#3).
        "fc1",
        "fc2",
        # Vision tower (DaViT) so the model learns to SEE notes better, not just to write
        # tokens (#1). "qkv" is unique to the DaViT attention blocks; the "fc1"/"fc2" above
        # also match the DaViT MLPs. We deliberately do NOT target bare "proj": it collides
        # with the DaViT patch-embed Conv2d and matches "out_proj" by suffix.
        "qkv",
        "lm_head",
    ],
)

# NOTE: get_peft_model() has no `ensure_weight_tying` argument (that raises TypeError).
# Florence-2 ties lm_head to the input embeddings; a LoRA adapter on lm_head adds a
# separate low-rank delta and does NOT modify the tied base weight, so the tie is
# preserved automatically.
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

# A100 throughput settings
model.config.use_cache = False
# The A100 plans reserve enough memory to avoid activation recomputation.
if GRADIENT_CHECKPOINTING:
    model.enable_input_require_grads()
    model.gradient_checkpointing_enable()
    print("Gradient checkpointing: ON")
else:
    print("Gradient checkpointing: OFF (A100 fast path)")

collate_fn = make_collate_fn(processor, TASK_PROMPT, MAX_LABEL_LENGTH)


In [ ]:
# @title 7) Train (resume-safe)
import glob

report_to = []
if USE_WANDB:
    report_to.append("wandb")
    os.environ.setdefault("WANDB_PROJECT", WANDB_PROJECT)

steps_per_epoch = max(1, (len(train_ds) + PER_DEVICE_BATCH_SIZE * GRAD_ACCUM_STEPS - 1) // (PER_DEVICE_BATCH_SIZE * GRAD_ACCUM_STEPS))
print(f"~{steps_per_epoch:,} optimizer steps / epoch")


def find_latest_checkpoint() -> str | None:
    ckpts = sorted(glob.glob(str(CHECKPOINT_DIR / "checkpoint-*")), key=lambda p: int(p.rsplit("-", 1)[-1]))
    return ckpts[-1] if ckpts else None


resume_path = None
if RESUME_FROM_CHECKPOINT:
    resume_path = find_latest_checkpoint()
    if resume_path:
        print(f"Resuming from {resume_path}")

# Reset any stale compiled wrapper, then re-enable Inductor with CUDA Graphs disabled.
# This preserves kernel fusion without reusing LoRA outputs between accumulation steps.
os.environ.pop("TORCHDYNAMO_DISABLE", None)
os.environ.pop("ACCELERATE_DYNAMO_BACKEND", None)
import torch._dynamo
import torch._inductor.config as inductor_config
torch._dynamo.reset()
torch._dynamo.config.disable = False
if hasattr(inductor_config.triton, "cudagraphs"):
    inductor_config.triton.cudagraphs = False
if hasattr(inductor_config.triton, "cudagraph_trees"):
    inductor_config.triton.cudagraph_trees = False
if hasattr(model, "_orig_mod"):
    model = model._orig_mod
print(
    "TorchDynamo enabled; CUDA Graphs:",
    getattr(inductor_config.triton, "cudagraphs", "unavailable"),
)


training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    logging_steps=LOGGING_STEPS,
    eval_strategy="no",
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    bf16=USE_BF16,
    fp16=USE_FP16,
    tf32=USE_BF16,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
    optim="adamw_torch_fused",
    torch_compile=TORCH_COMPILE,
    torch_compile_backend="inductor",
    # Do not use reduce-overhead: it opts into CUDA Graph capture.
    torch_compile_mode="default",
    remove_unused_columns=False,
    dataloader_drop_last=True,
    dataloader_num_workers=DATALOADER_WORKERS,
    dataloader_pin_memory=(DEVICE == "cuda"),
    dataloader_persistent_workers=(DATALOADER_WORKERS > 0),
    dataloader_prefetch_factor=(6 if DATALOADER_WORKERS > 0 else None),
    report_to=report_to,
    seed=SEED,
)

class GoldenEpochEvalCallback(TrainerCallback):
    """Run the 100-image golden benchmark after each full training epoch."""

    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        if not EVAL_GOLDEN_EVERY_EPOCH:
            return
        if model is None:
            print("Golden epoch eval skipped — no model in callback kwargs")
            return

        epoch_num = int(round(float(state.epoch)))
        if epoch_num < 1:
            return

        print(f"\n=== Golden eval after epoch {epoch_num} ===")
        was_training = model.training
        try:
            results = run_golden_eval(
                model,
                processor,
                golden_records,
                phase=f"after_epoch_{epoch_num}",
                model_id=MODEL_ID,
                task_prompt=TASK_PROMPT,
                device=DEVICE,
                use_lora=True,
            )
            print_golden_eval_summary(results, f"After epoch {epoch_num}")

            baseline = globals().get("baseline_results")
            if baseline is None and EVAL_GOLDEN_BEFORE_PATH.is_file():
                baseline = json.loads(
                    EVAL_GOLDEN_BEFORE_PATH.read_text(encoding="utf-8")
                )
            if baseline is not None:
                compare_golden_eval_results(baseline, results)

            save_golden_epoch_eval(results, epoch=epoch_num, baseline=baseline)
        finally:
            if was_training:
                model.train()
            if DEVICE == "cuda":
                torch.cuda.empty_cache()


class StyleBalancedTrainer(Trainer):
    """WeightedRandomSampler so each epoch sees ~50/25/25 handwritten/clean/camera."""

    def __init__(self, *args, train_sample_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self._train_sample_weights = train_sample_weights

    def _get_train_sampler(self, train_dataset=None):
        # Newer transformers pass the dataset; older ones call with no args.
        dataset = self.train_dataset if train_dataset is None else train_dataset
        if self._train_sample_weights is not None and dataset is self.train_dataset:
            from torch.utils.data import WeightedRandomSampler

            return WeightedRandomSampler(
                weights=self._train_sample_weights,
                num_samples=len(dataset),
                replacement=True,
            )
        try:
            return super()._get_train_sampler(train_dataset)
        except TypeError:
            return super()._get_train_sampler()


trainer = StyleBalancedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    data_collator=collate_fn,
    callbacks=[GoldenEpochEvalCallback()],
    train_sample_weights=train_sample_weights,
)

train_result = trainer.train(resume_from_checkpoint=resume_path)
print(train_result)

# Save latest checkpoint adapter snapshot (no random val holdout)
best_dir = DRIVE_OUTPUT_DIR / "best"
latest_ckpt = find_latest_checkpoint()
if latest_ckpt:
    print(f"Copying latest checkpoint -> {best_dir}")
    if best_dir.exists():
        shutil.rmtree(best_dir)
    shutil.copytree(latest_ckpt, best_dir)
    processor.save_pretrained(str(best_dir))
else:
    trainer.save_model(str(best_dir))
    processor.save_pretrained(str(best_dir))
print(f"Saved adapter snapshot -> {best_dir}")


## Post-training eval

Training already runs the same 100-image golden benchmark **after every full epoch** (see `eval_golden_epochs/` + `eval_golden_epochs.md` on Drive).

This cell re-runs the benchmark on the final LoRA weights, writes `eval_golden_after.json`, and compares against the baseline.


In [ ]:
# @title 8) Post-training golden benchmark (fine-tuned LoRA)
after_results = run_golden_eval(
    model,
    processor,
    golden_records,
    phase="after_training",
    model_id=MODEL_ID,
    task_prompt=TASK_PROMPT,
    device=DEVICE,
    use_lora=True,
)
print_golden_eval_summary(after_results, "After training (LoRA)")
EVAL_GOLDEN_AFTER_PATH.write_text(json.dumps(after_results, indent=2), encoding="utf-8")
print(f"Wrote {EVAL_GOLDEN_AFTER_PATH}")

if baseline_results is None and EVAL_GOLDEN_BEFORE_PATH.is_file():
    baseline_results = json.loads(EVAL_GOLDEN_BEFORE_PATH.read_text(encoding="utf-8"))

if baseline_results is not None:
    compare_golden_eval_results(baseline_results, after_results)
else:
    print("No baseline results available for comparison")

In [ ]:
# @title 9) Export merged model to Drive (+ optional Hugging Face Hub)
final_dir = DRIVE_OUTPUT_DIR / "final"
final_dir.mkdir(parents=True, exist_ok=True)

# merge_and_unload() fuses LoRA weights into the base model for simpler inference
merged = trainer.model.merge_and_unload()
merged.save_pretrained(str(final_dir))
processor.save_pretrained(str(final_dir))
print(f"Merged model saved -> {final_dir}")

if HF_REPO_ID:
    if not HF_TOKEN:
        print("Set HF_TOKEN to push to the Hub.")
    else:
        merged.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
        processor.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
        print(f"Pushed to https://huggingface.co/{HF_REPO_ID}")

## Multi-session resume

A full Florence-2-large run is expected to take roughly 6–9 hours based on the measured 2.5-hour base-model run. Drive checkpoints protect against runtime interruption.

1. Ensure checkpoints are on Drive: `MyDrive/omr/florence2-large-handnew-r64-lr1e4/checkpoints/checkpoint-*`
2. Re-open the notebook, run the setup cells, then training with `RESUME_FROM_CHECKPOINT = True`
3. Set `SKIP_BASELINE_EVAL = True` if `eval_golden_before.json` already exists on Drive
4. The Trainer restores model, optimizer, scheduler, and step state from the latest checkpoint
5. Per-epoch golden results persist in `eval_golden_epochs/` and `eval_golden_epochs.md`

To **force a fresh run**, delete `checkpoints/` and `manifest.json` on Drive (only if you changed split settings).

## Local path reference

| Repo path | Purpose |
|-----------|---------|
| `data/consolidated/images/` | ~139k training images (`{group}__{id}.png/.jpg`) |
| `data/consolidated/ground_truth/` | Matching `.semantic` files |
| `golden_set/` | 100-image OMR benchmark (upload full folder for eval) |
| `golden_set/benchmark/selection_*.json` | Optional; overrides inline benchmark list |
| `scripts/omr_benchmark_run_vlm.py` | Zero-shot Florence-2 inference baseline |
| `scripts/omr_benchmark_score.py` | Token-level Levenshtein scoring reference |
